In [19]:
import xarray as xr
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr, wasserstein_distance
from skimage.metrics import structural_similarity as ssim
from sklearn.metrics import classification_report, jaccard_score, cohen_kappa_score, confusion_matrix
from scipy import stats
import pandas as pd

In [20]:
target_res = 0.125
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.5, -60.0, -44.875]
min_time, max_time = pd.to_datetime("2013-01-01"), pd.to_datetime("2025-12-31")
split_year_test = 2025

sp = "SQA"
fishing_ds = xr.open_dataset(f"../data/processed/targets/cpue_{target_res}.nc").sel(FAOspp=sp)
fishing = fishing_ds["cpue_index"].fillna(0)

fishing_class_ds = xr.open_dataset(f"../data/processed/targets/cpue_{target_res}.nc").sel(FAOspp=sp)
fishing_class = fishing_class_ds["CPUE_class"].fillna(0)

mask_ds = xr.open_dataset(f"../data/processed/static/area_pesca_{target_res}.nc")
mask = mask_ds["mask"].fillna(0)
mask = mask.broadcast_like(fishing)

fishing, mask, fishing_class = xr.align(fishing, mask, fishing_class, join="inner")

#area de pesca y un poco alrededor
croppedT = lambda da: da.sel(
    lon=slice(min_lon-0.5, max_lon+0.5),
    lat=slice(min_lat-0.5, max_lat+0.5),
    time=slice(min_time, max_time)
)
fishing_cropped = croppedT(fishing)
mask_cropped = croppedT(mask)
fishing_class_cropped = croppedT(fishing_class)

print(fishing_cropped.shape)
print(mask_cropped.shape)
print(fishing_class_cropped.shape)


(156, 30, 17)
(156, 30, 17)
(156, 30, 17)


In [21]:
#regressions
fishing_train = fishing_cropped.sel(time=slice(None, f"{split_year_test-1}-12-31"))
fishing_test = fishing_cropped.sel(time=slice(f"{split_year_test}-01-01", None))

mask_train = mask_cropped.sel(time=slice(None, f"{split_year_test-1}-12-31"))
mask_test = mask_cropped.sel(time=slice(f"{split_year_test}-01-01", None))

H_out_test, W_out_test = fishing_test.shape[1], fishing_test.shape[2]

In [22]:
def compute_baseline_metrics_regression(y_true_da, y_pred_da, mask_da):
    # =====================================================
    # 1. EXTRACT ARRAYS & FLATTEN FOR METRICS
    # =====================================================
    # Extract underlying numpy arrays
    y_true = y_true_da.values
    y_pred = y_pred_da.values
    mask_spatial = mask_da.values > 0  # Ensure boolean
    
    # Broadcast 2D mask to 3D time series if necessary
    if mask_spatial.ndim == 2:
        mask_spatial = np.broadcast_to(mask_spatial, y_true.shape)

    # Flatten for global sklearn metrics
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    mask_flat = mask_spatial.flatten()
    
    y_test_masked = y_true_flat[mask_flat]
    y_pred_masked = y_pred_flat[mask_flat]

    # Clean NaNs strictly for valid indices (just in case they slipped through)
    valid = ~np.isnan(y_test_masked) & ~np.isnan(y_pred_masked)
    y_test_masked = y_test_masked[valid]
    y_pred_masked = y_pred_masked[valid]

    # =====================================================
    # 2. REGRESSION METRICS
    # =====================================================
    rmse = np.sqrt(mean_squared_error(y_test_masked, y_pred_masked))
    mae = mean_absolute_error(y_test_masked, y_pred_masked)
    r2 = r2_score(y_test_masked, y_pred_masked)
    pearson_corr = pearsonr(y_test_masked, y_pred_masked)[0]
    mbe = np.mean(y_pred_masked - y_test_masked)
    data_range_flat = np.max(y_test_masked) - np.min(y_test_masked)
    nrmse = rmse / data_range_flat if data_range_flat > 0 else 0

    w_dist = wasserstein_distance(y_test_masked, y_pred_masked)
    spearman_corr, p_val = spearmanr(y_test_masked, y_pred_masked)

    print("RMSE:", f"{rmse:.4f}")
    print(f"Normalized RMSE: {nrmse:.4f} ({nrmse*100:.2f}%)")
    print("MAE:", f"{mae:.4f}")
    print("R2:", f"{r2:.4f}")
    print("Pearson R:", f"{pearson_corr:.4f}")
    print("MBE:", f"{mbe:.4f}")
    print(f"Wasserstein Distance: {w_dist:.4f}")
    print(f"Spearman Rank Correlation: {spearman_corr:.2f}")

   

    
    metrics = [r2, pearson_corr, rmse, nrmse, mae, mbe, w_dist, spearman_corr]
    columns = ["r2", "pearson_corr", "rmse", "nrmse", "mae", "mbe", "w_dist", "spearman_corr"]
    metrics_df = pd.DataFrame(columns=columns)
    metrics_df.loc[len(metrics_df)] = metrics

    return metrics_df

In [23]:

climatology = fishing_train.groupby("time.month").mean("time")
fishing_pred = climatology.sel(month=fishing_test.time.dt.month).drop_vars("month")
metrics_df= compute_baseline_metrics_regression(fishing_test, fishing_pred, mask_test)




sheet_name = f"Climatology_{sp}"
with pd.ExcelWriter("./modelos/resultados/Regre/BL_reg.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    metrics_df.to_excel(writer, sheet_name=sheet_name, startrow=1, index=False)

########## Save the prediction to NetCDF ###########
mask_flat = mask_test.astype(bool)

y_pred_masked_flat = np.where(mask_flat, fishing_pred, np.nan)
y_true_masked_flat = np.where(mask_flat, fishing_pred, np.nan)

# Reshape back to spatial maps
all_preds = y_pred_masked_flat.reshape(-1, H_out_test, W_out_test)
all_targets = y_true_masked_flat.reshape(-1, H_out_test, W_out_test)

# Coordinates
lats = fishing_cropped["lat"].values
lons = fishing_cropped["lon"].values

# Safer: generate exactly as many dates as predictions
times = pd.date_range(
    start="2025-01-01",
    periods=all_preds.shape[0],
    freq="MS"
)

# Build xarray Dataset
ds_test = xr.Dataset(
    {
        "pred": (("time", "lat", "lon"), all_preds),
        "target": (("time", "lat", "lon"), all_targets),
    },
    coords={
        "time": times,
        "lat": lats,
        "lon": lons,
    }
)

# Save to NetCDF
ds_test.to_netcdf(f"./prueba_modelo/predicted/predicted_BL_{sp}.nc", mode="w")



RMSE: 1.2890
Normalized RMSE: 0.1121 (11.21%)
MAE: 0.7105
R2: 0.4711
Pearson R: 0.6927
MBE: -0.1285
Wasserstein Distance: 0.4375
Spearman Rank Correlation: 0.65


In [24]:

fishing_class_train = fishing_class_cropped.sel(time=slice(None, f"{split_year_test-1}-12-31"))
fishing_class_test = fishing_class_cropped.sel(time=slice(f"{split_year_test}-01-01", None))

def get_mode(x, axis, mode=True):
    if mode:
        return stats.mode(x, axis=axis, keepdims=False).mode
    else:
        return np.mean(x, axis=axis).round()

climatology_class = fishing_class_train.groupby("time.month").reduce(get_mode, mode=True)

# 4. Predict on the Test Period
fishing_class_pred = climatology_class.sel(month=fishing_class_test.time.dt.month).drop_vars("month")


In [25]:
def compute_climatology_metrics_classification(y_true_da, y_pred_da, mask_da):
    # =====================================================
    # 1. EXTRACT ARRAYS & FLATTEN FOR METRICS
    # =====================================================
    # Extract underlying numpy arrays
    y_true = y_true_da.values
    y_pred = y_pred_da.values
    mask_spatial = mask_da.values > 0  # Ensure boolean
    
    # Broadcast 2D mask to 3D time series if necessary
    if mask_spatial.ndim == 2:
        mask_spatial = np.broadcast_to(mask_spatial, y_true.shape)

    # Flatten for global sklearn metrics
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    mask_flat = mask_spatial.flatten()
    
    y_test_masked = y_true_flat[mask_flat]
    y_pred_masked = y_pred_flat[mask_flat]

    # Clean NaNs strictly for valid indices (just in case they slipped through)
    valid = ~np.isnan(y_test_masked) & ~np.isnan(y_pred_masked)
    y_test_masked = y_test_masked[valid]
    y_pred_masked = y_pred_masked[valid]

    # =====================================================
    # 2. Classification Metrics
    # =====================================================
    print("=== Classification Report ===")
    report = pd.DataFrame(classification_report(y_test_masked, y_pred_masked, labels=[0, 1, 2], output_dict=True)).T
    print(report)
    
    iou_per_class = jaccard_score(y_test_masked, y_pred_masked, average=None, labels=[0, 1, 2])
    kappa = cohen_kappa_score(y_test_masked, y_pred_masked)
    
    print("=== Spatial Segmentation Metrics ===")
    print(f"IoU Class 1 (Low CPUE):      {iou_per_class[1]:.3f}")
    print(f"IoU Class 2 (High CPUE):     {iou_per_class[2]:.3f}")
    print(f"Cohen's Kappa Score:         {kappa:.3f}\n")
    
    # New Plot: Confusion Matrix
    class_labels = ['None (0)', 'Low CPUE (1)', 'High CPUE (2)']
    cm = pd.DataFrame(confusion_matrix(y_test_masked,y_pred_masked,labels=[0, 1, 2]),index=class_labels,columns=class_labels)

    # =====================================================
    # 5. SSIM (Structural Similarity)
    # =====================================================
    # Replace NaNs with 0s globally for map functions (SSIM and Plotting)
    y_true_clean = np.nan_to_num(y_true, nan=0.0)
    y_pred_clean = np.nan_to_num(y_pred, nan=0.0)
    
    masked_data = y_true_clean[mask_spatial]
    data_range = masked_data.max() - masked_data.min() if len(masked_data) > 0 else 1.0

    ssim_scores = []
    for i in range(y_true.shape[0]):
        _, ssim_map = ssim(
            y_true_clean[i],
            y_pred_clean[i],
            data_range=data_range,
            win_size=5,
            full=True
        )
        valid_ssim_pixels = ssim_map[mask_spatial[i]]
        if len(valid_ssim_pixels) > 0:
            ssim_scores.append(np.mean(valid_ssim_pixels))

    mean_ssim = np.mean(ssim_scores) if ssim_scores else 0
    print(f"Mean Masked SSIM: {mean_ssim:.4f}\n")



    return report, iou_per_class, kappa, cm

In [26]:
report, iou_per_class, kappa, cm = compute_climatology_metrics_classification(fishing_class_test, fishing_class_pred, mask_test)

iou_per_class = pd.DataFrame(iou_per_class, index=["Class 0", "Class 1", "Class 2"], columns=["IoU"])

metrics_df = pd.DataFrame({
    "Cohen's Kappa": [kappa],
})

sheet_name = f"Climatology_{sp}"
# save everyhtng to a excel workbook
with pd.ExcelWriter("./modelos/resultados/Class/BL_cla.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    report.to_excel(writer, sheet_name=sheet_name, startrow=1)

with pd.ExcelWriter("./modelos/resultados/Class/BL_cla.xlsx", engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
    iou_per_class.to_excel(writer, sheet_name=sheet_name, startrow=len(report) + 3)
    metrics_df.to_excel(writer, sheet_name=sheet_name, startrow=len(report) + len(iou_per_class) + 6, index=False)



=== Classification Report ===
              precision    recall  f1-score      support
0              0.768574  0.974026  0.859189   924.000000
1              0.777778  0.437500  0.560000   288.000000
2              0.868263  0.503472  0.637363   288.000000
accuracy       0.780667  0.780667  0.780667     0.780667
macro avg      0.804872  0.638333  0.685517  1500.000000
weighted avg   0.789481  0.780667  0.759154  1500.000000
=== Spatial Segmentation Metrics ===
IoU Class 1 (Low CPUE):      0.389
IoU Class 2 (High CPUE):     0.468
Cohen's Kappa Score:         0.540

Mean Masked SSIM: 0.4102



In [27]:
#Short term classification climatology
recent_fishing_class_train = fishing_class_train.sel(time=slice("2019-01-01", None))
climatology_class = recent_fishing_class_train.groupby("time.month").reduce(get_mode)

# 4. Predict on the Test Period
fishing_class_pred = climatology_class.sel(month=fishing_class_test.time.dt.month).drop_vars("month")

report, iou_per_class, kappa, cm = compute_climatology_metrics_classification(fishing_class_test, fishing_class_pred, mask_test)


=== Classification Report ===
              precision    recall  f1-score      support
0              0.774390  0.962121  0.858108   924.000000
1              0.715909  0.437500  0.543103   288.000000
2              0.818182  0.500000  0.620690   288.000000
accuracy       0.772667  0.772667  0.772667     0.772667
macro avg      0.769494  0.633207  0.673967  1500.000000
weighted avg   0.771570  0.772667  0.752043  1500.000000
=== Spatial Segmentation Metrics ===
IoU Class 1 (Low CPUE):      0.373
IoU Class 2 (High CPUE):     0.450
Cohen's Kappa Score:         0.530

Mean Masked SSIM: 0.4191

